# 03b — Series Feature Engineering

Aggregate game data to **one row per playoff series** and build pre-series features.

**Input:** `data/raw/playoff_games_*.parquet`, `data/raw/team_metrics_*.parquet`  
**Output:** `data/processed/series_features.parquet`

**Target:** `higher_seed_wins` — 1 if the home-court team (higher seed) wins the series

**Why series?** Individual game outcomes have ~40% irreducible variance. Series outcomes average over 4–7 games, making team quality a much stronger signal.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.insert(0, str(Path().resolve().parent))

In [ ]:
# CONFIG
RAW_DIR = Path().resolve().parent / "data" / "raw"
PROCESSED_DIR = Path().resolve().parent / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_PATH = PROCESSED_DIR / "series_features.parquet"

SEASONS = [
    "2014-15",
    "2015-16",
    "2016-17",
    "2017-18",
    "2018-19",
    "2020-21",
    "2021-22",
    "2022-23",
    "2023-24",
]

## 1. Load Raw Data

In [ ]:
games = pd.concat(
    [
        pd.read_parquet(RAW_DIR / f"playoff_games_{s}.parquet").assign(season=s)
        for s in SEASONS
    ],
    ignore_index=True,
)
metrics = pd.concat(
    [
        pd.read_parquet(RAW_DIR / f"team_metrics_{s}.parquet").assign(season=s)
        for s in SEASONS
    ],
    ignore_index=True,
)

games["round"] = games["GAME_ID"].str[6:8].astype(int)
games["GAME_DATE"] = pd.to_datetime(games["GAME_DATE"])

# Home-team rows only (one row per game)
home = games[games["MATCHUP"].str.contains(r"vs\.")].copy()
away = games[games["MATCHUP"].str.contains("@")][["GAME_ID", "TEAM_ID"]].rename(
    columns={"TEAM_ID": "away_team_id"}
)
game_df = home.merge(away, on="GAME_ID", how="inner")
game_df["home_win"] = (game_df["WL"] == "W").astype(int)
game_df = game_df.rename(columns={"TEAM_ID": "home_team_id"})

# Consistent series key
game_df["series_key"] = game_df.apply(
    lambda r: (
        f"{r['season']}_{r['round']}_{min(r['home_team_id'], r['away_team_id'])}_{max(r['home_team_id'], r['away_team_id'])}"
    ),
    axis=1,
)

print(f"{len(game_df)} games across {game_df['series_key'].nunique()} series")

## 2. Aggregate to One Row Per Series

In [ ]:
# Game 1 home team = higher seed (hosts games 1, 2, 5, 7)
game1 = (
    game_df.sort_values("GAME_DATE")
    .groupby("series_key")
    .first()[["home_team_id", "away_team_id"]]
    .reset_index()
    .rename(columns={"home_team_id": "higher_seed_id", "away_team_id": "lower_seed_id"})
)

# Tag each game with whether the higher seed won (regardless of venue)
game_df = game_df.merge(game1, on="series_key")
game_df["higher_seed_won_game"] = (
    ((game_df["home_team_id"] == game_df["higher_seed_id"]) & (game_df["WL"] == "W"))
    | ((game_df["home_team_id"] != game_df["higher_seed_id"]) & (game_df["WL"] == "L"))
).astype(int)

# Aggregate to one row per series
series_agg = (
    game_df.sort_values("GAME_DATE")
    .groupby("series_key")
    .agg(
        season=("season", "first"),
        round=("round", "first"),
        home_team_id=("higher_seed_id", "first"),
        away_team_id=("lower_seed_id", "first"),
        series_length=("higher_seed_won_game", "count"),
        higher_seed_game_wins=("higher_seed_won_game", "sum"),
        series_start_date=("GAME_DATE", "min"),
    )
    .reset_index()
)

series_agg["lower_seed_game_wins"] = (
    series_agg["series_length"] - series_agg["higher_seed_game_wins"]
)
series_agg["higher_seed_wins"] = (
    series_agg["higher_seed_game_wins"] > series_agg["lower_seed_game_wins"]
).astype(int)

print(f"{len(series_agg)} series total")
print(
    f"Higher seed wins rate: {series_agg['higher_seed_wins'].mean():.1%}  (expect ~65%)"
)
print("\nSeries length distribution:")
print(series_agg["series_length"].value_counts().sort_index().to_string())
print("\nSeries per round:")
print(
    series_agg["round"]
    .value_counts()
    .sort_index()
    .rename({1: "R1", 2: "R2", 3: "Conf Finals", 4: "Finals"})
    .to_string()
)

## 3. Merge Team Efficiency Features

In [ ]:
eff = metrics[
    [
        "TEAM_ID",
        "season",
        "E_OFF_RATING",
        "E_DEF_RATING",
        "E_NET_RATING",
        "E_PACE",
        "W_PCT",
        "E_OREB_PCT",
        "E_TM_TOV_PCT",
    ]
].copy()

df = series_agg.merge(
    eff.rename(
        columns={
            "TEAM_ID": "home_team_id",
            "E_OFF_RATING": "home_ortg",
            "E_DEF_RATING": "home_drtg",
            "E_NET_RATING": "home_net_rtg",
            "E_PACE": "home_pace",
            "W_PCT": "home_win_pct",
            "E_OREB_PCT": "home_oreb_pct",
            "E_TM_TOV_PCT": "home_tov_pct",
        }
    ),
    on=["home_team_id", "season"],
    how="left",
)
df = df.merge(
    eff.rename(
        columns={
            "TEAM_ID": "away_team_id",
            "E_OFF_RATING": "away_ortg",
            "E_DEF_RATING": "away_drtg",
            "E_NET_RATING": "away_net_rtg",
            "E_PACE": "away_pace",
            "W_PCT": "away_win_pct",
            "E_OREB_PCT": "away_oreb_pct",
            "E_TM_TOV_PCT": "away_tov_pct",
        }
    ),
    on=["away_team_id", "season"],
    how="left",
)

# Differentials (positive = home/higher-seed advantage)
df["ortg_diff"] = df["home_ortg"] - df["away_ortg"]
df["drtg_diff"] = df["home_drtg"] - df["away_drtg"]
df["net_rtg_diff"] = df["home_net_rtg"] - df["away_net_rtg"]
df["win_pct_diff"] = df["home_win_pct"] - df["away_win_pct"]

print(f"Missing efficiency: {df[['home_ortg', 'away_ortg']].isnull().sum().to_dict()}")

## 4. Historical Playoff Features

In [ ]:
# Build season-level playoff stats for historical features
playoff_hist = games.copy()
playoff_hist["win"] = (playoff_hist["WL"] == "W").astype(int)
playoff_hist["is_finals"] = (playoff_hist["round"] == 4).astype(int)

season_stats = (
    playoff_hist.groupby(["season", "TEAM_ID"])
    .agg(
        playoff_wins=("win", "sum"),
        playoff_games=("win", "count"),
        reached_finals=("is_finals", "max"),
    )
    .reset_index()
)
season_stats["playoff_win_pct"] = (
    season_stats["playoff_wins"] / season_stats["playoff_games"]
)
season_stats["season_year"] = season_stats["season"].str[:4].astype(int)


def prior_stats(team_id, current_season, stats_df):
    yr = int(current_season[:4])
    hist = stats_df[(stats_df["TEAM_ID"] == team_id) & (stats_df["season_year"] < yr)]
    p3 = hist[hist["season_year"] >= yr - 3]
    p5 = hist[hist["season_year"] >= yr - 5]
    return (
        p3["playoff_win_pct"].mean() if len(p3) > 0 else np.nan,
        int(p5["reached_finals"].sum()) if len(p5) > 0 else 0,
    )


home_hist = df.apply(
    lambda r: prior_stats(r["home_team_id"], r["season"], season_stats), axis=1
)
away_hist = df.apply(
    lambda r: prior_stats(r["away_team_id"], r["season"], season_stats), axis=1
)

df["home_playoff_win_pct_3yr"] = [x[0] for x in home_hist]
df["away_playoff_win_pct_3yr"] = [x[0] for x in away_hist]
df["home_finals_apps_5yr"] = [x[1] for x in home_hist]
df["away_finals_apps_5yr"] = [x[1] for x in away_hist]

df["home_playoff_win_pct_3yr"] = df["home_playoff_win_pct_3yr"].fillna(0.5)
df["away_playoff_win_pct_3yr"] = df["away_playoff_win_pct_3yr"].fillna(0.5)
df["playoff_win_pct_diff"] = (
    df["home_playoff_win_pct_3yr"] - df["away_playoff_win_pct_3yr"]
)

print("Historical feature summary:")
print(
    df[
        [
            "home_playoff_win_pct_3yr",
            "away_playoff_win_pct_3yr",
            "home_finals_apps_5yr",
            "away_finals_apps_5yr",
        ]
    ]
    .describe()
    .round(3)
)

## 5. Assemble + Save

In [ ]:
FEATURE_COLS = [
    # Efficiency
    "home_ortg",
    "away_ortg",
    "home_drtg",
    "away_drtg",
    "home_net_rtg",
    "away_net_rtg",
    "net_rtg_diff",
    "home_pace",
    "away_pace",
    "ortg_diff",
    "drtg_diff",
    # Team quality
    "home_win_pct",
    "away_win_pct",
    "win_pct_diff",
    "home_oreb_pct",
    "away_oreb_pct",
    "home_tov_pct",
    "away_tov_pct",
    # Historical
    "home_playoff_win_pct_3yr",
    "away_playoff_win_pct_3yr",
    "playoff_win_pct_diff",
    "home_finals_apps_5yr",
    "away_finals_apps_5yr",
]
META_COLS = [
    "series_key",
    "season",
    "round",
    "home_team_id",
    "away_team_id",
    "series_length",
    "higher_seed_game_wins",
    "lower_seed_game_wins",
]
TARGET_COL = "higher_seed_wins"

out = df[META_COLS + FEATURE_COLS + [TARGET_COL]].copy()

assert out[TARGET_COL].isnull().sum() == 0
assert out[FEATURE_COLS].isnull().sum().sum() == 0

out.to_parquet(OUTPUT_PATH, index=False)
print(f"Saved {len(out)} series to {OUTPUT_PATH}")
print(f"Features: {len(FEATURE_COLS)}")
print(f"Higher seed win rate: {out[TARGET_COL].mean():.1%}")
print("\nSeries per season:")
print(out.groupby("season").size().to_string())